In [1]:
# google colab setup
# !pip install -q huggingface_hub fsspec


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

import utils
RANDOM_SEED = 23
np.random.seed(RANDOM_SEED)


In [3]:
df = pd.read_csv("./dataset/diabetes_binary_health_indicators_BRFSS2015.csv")


In [4]:
# checking the balance of the class
utils.get_class_distribution_info(df, 'Diabetes_binary')


Class Distribution:
Class 0: 218334 samples (86.07%)
Class 1: 35346 samples (13.93%)


. The imblearn Module
imblearn (imbalanced-learn) is an official extension of the scikit-learn ecosystem. It provides specific algorithms (like SMOTE) and tools (like an upgraded Pipeline) to handle datasets where one target class heavily outnumbers the other.

2. StandardScaler
Machine learning models struggle when features are measured on different scales (e.g., BMI ranges from 15-50, but a binary health flag is 0 or 1).

What it does: It standardizes features by shifting the mean to 0 and scaling the variance to 1.

Why it matters: It ensures no single feature artificially dominates the model just because its raw numbers are larger.

3. Over-sampling and SMOTE
Over-sampling: The process of balancing a dataset by increasing the number of minority class samples (in your case, the diabetic patients) to match the majority class (healthy patients).

What SMOTE is: SMOTE (Synthetic Minority Over-sampling Technique) does not simply duplicate existing rows. It uses K-Nearest Neighbors mathematics to generate entirely new, synthetic data points that safely blend the characteristics of existing minority class patients.

In [5]:
# splitting the data as the first step, to avoid data leakage
X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

# use 20% of the dataset as test set
# using stratify=y, to ensure the split will address the class imbalance
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)


pipeline = Pipeline(
	[
		('scalar', StandardScaler()),
		('smote', SMOTE(random_state=RANDOM_SEED)),
		('classifier', RandomForestClassifier(
			n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
		)
	]
)


In [6]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print('Running 5-fold cross-validation...')
cv_scores = cross_val_score(
	pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f'ROC-AUC scores for each fold: {cv_scores}')
print(f'Average CV ROC-AUC score: {cv_scores.mean():.2f} ± {cv_scores.std()*2:.2f}')



Running 5-fold cross-validation...
ROC-AUC scores for each fold: [0.80074122 0.7958633  0.79829276 0.79972277 0.79567113]
Average CV ROC-AUC score: 0.80 ± 0.00


In [ ]:
import shap

print('Fitting final pipeline...')
pipeline.fit(X_train, y_train)

# extracting model and scalar from the pipeline
scalar = pipeline.named_steps['scalar']
model = pipeline.named_steps['classifier']

X_test_scaled = pd.DataFrame(scalar.transform(X_test), columns=X.columns)
X_test_sample = shap.sample(X_test_scaled, nsamples=1000)

print('Calculating SHAP values...')
explainer = shap.TreeExplainer(model=model)
shap_values = explainer(X_test_sample)

shap.plots.beeswarm(shap_values[:, :, 1], max_display=10)


Fitting final pipeline...
Calculating SHAP values...
